# DepChainTagger

This notebook shows how to configure `DepChainTagger`, how to define a `PathPattern`, and how to run the tagger on a sample text. It also explains the most important parts of the output so you can adapt the example to your own texts.


In [1]:
import estnltk

from scripts.DepChainTagger import (
    ConditionMode,
    DepChainTagger,
    DepTaggerOrchestrator,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    ValueCondition,
    NestedValueCondition,
    SyntaxGraphIndex,
)

from estnltk_neural.taggers import StanzaSyntaxTagger

e:\Git_projects\EstNLTK\EstNLTK_DependencyChains\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tagger parameters

The current `DepChainTagger` constructor focuses on output control and match filtering. The tagger always reads from the `stanza_syntax` and `sentences` layers internally, and it writes a relation layer named `dep_chains` by default.

| Parameter                     | What it controls                                                                        |
| ----------------------------- | --------------------------------------------------------------------------------------- |
| **`patterns`**                | A tuple of `PathPattern` objects that define what the tagger should match.              |
| `output_layer`                | Name of the relation layer created by the tagger.                                       |
| `output_attributes`           | Extra attributes written to the output layer. If omitted, the package default is used.  |
| `include_pattern_constraints` | Whether to include the pattern constraints as attributes in the output spans/relations. |
| `sentence_match_dedup_mode`   | Deduplication inside one sentence: `none`, `exact`, or `role_based`.                    |
| `max_matches_per_sentence`    | Maximum number of matches to keep per sentence.                                         |
| `allow_role_node_overlap`     | Whether the same node may fill more than one role in a pattern.                         |
| `global_dedup_mode`           | Deduplication across the whole text: `none`, `exact`, or `role_based`.                  |
| `max_total_matches`           | Upper limit for all matches across the entire text.                                     |

Everything, except for the **`patterns`**, is optional and has a default value. The deduplication modes and their implications are explained in more detail below.


## Preparing a sample text

The tagger expects a text object with sentence segmentation and a syntax layer.


In [2]:
# Download the Stanza models for Estonian if not already present
stanza_syntax_tagger = StanzaSyntaxTagger(
    input_type="morph_extended",
    input_morph_layer="morph_extended",
    add_parent_and_children=True,
)
# estnltk.download("stanzasyntaxtagger")

In [3]:
# sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
# sample_text = "Üks ütles, et 1. mail tähistab palju maid töörahvapüha."

In [4]:
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer(
    "morph_extended"
)  # StanzaSyntaxTagger expects morph layer to be present
stanza_syntax_tagger.tag(text_obj)

Text(text="1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.")

Let's look at the syntax layer to see what the syntax tagger has produced for our sample sentence.


In [5]:
display(text_obj.stanza_syntax)

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc', 'parent_span', 'children'), spans=SL[Span('1990.', [{'id': 1, 'lemma': '1990.', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict({'ord': 'ord', '<?>': '<?>', 'roman': 'roman'}), 'head': 2, 'deprel': 'nummod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('aasta', [{'id': 2, 'lemma': 'aasta', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'com': 'com', 'sg': 'sg', 'gen': 'gen'}), 'head': 4, 'deprel': 'nmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('kuumal', [{'id': 3, 'lemma': 'kuum', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'com': 'com', 'sg': 'sg', 'ad': 'ad'}), 'head': 4, 'deprel': 'amod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('suvel', [{'id': 4, 'lemma': 'suvi', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'com': 'com', 'sg': 'sg', 'ad': 'ad'}), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('vaatas', [{'id': 5, 'lemma': 'vaatama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict({'mod': 'mod', 'indic': 'indic', 'impf': 'impf', 'ps3': 'ps3', 'sg': 'sg', 'ps': 'ps', 'af': 'af'}), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_', 'parent_span': None, 'children': <class 'tuple'>}]),
Span('Bureau', [{'id': 6, 'lemma': 'Bureau', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'prop': 'prop', 'sg': 'sg', 'nom': 'nom'}), 'head': 5, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('Veritas', [{'id': 7, 'lemma': 'Veritas', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'prop': 'prop', 'sg': 'sg', 'nom': 'nom'}), 'head': 6, 'deprel': 'flat', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span("Estline'i", [{'id': 8, 'lemma': 'Estline', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'prop': 'prop', 'sg': 'sg', 'gen': 'gen'}), 'head': 10, 'deprel': 'obl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('omanduseks', [{'id': 9, 'lemma': 'omandus', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'com': 'com', 'sg': 'sg', 'tr': 'tr'}), 'head': 10, 'deprel': 'obl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('saanud', [{'id': 10, 'lemma': 'saama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict({'mod': 'mod', 'indic': 'indic', 'impf': 'impf', 'ps': 'ps', 'neg': 'neg'}), 'head': 11, 'deprel': 'acl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('laeva', [{'id': 11, 'lemma': 'laev', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'com': 'com', 'sg': 'sg', 'part': 'part'}), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('uuesti', [{'id': 12, 'lemma': 'uuesti', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 5, 'deprel': 'advmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('üle', [{'id': 13, 'lemma': 'üle', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 5, 'deprel': 'compound:prt', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('.', [{'id': 14, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 5, 'deprel': 'punct', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}])])

## Constructing a tagger with a simple pattern


A tagger is configured by passing the desired parameters to the constructor. The most important parameter is `patterns`: a tuple of `PathPattern` objects that define what the tagger will look for. A `PathPattern` consists of:

- `node_steps` (a sequence of `NodeConstraint` objects) and
- `edge_steps` (a sequence of `EdgeConstraint` objects); in typical linear patterns `len(edge_steps) == len(node_steps) - 1`.

Each `NodeConstraint` assigns a `role` (this role name is used in the output relation) and can include attribute or feature conditions; each `EdgeConstraint` specifies direction, hop limits and optional attribute conditions. The matcher traverses the `stanza_syntax` tree following the pattern and emits matches as relations (default layer `dep_chains`) where roles point to the matched syntax nodes.

Below is a simplified overview of the workflow:

```
(NodeConstraints, EdgeConstraints) --> PathPattern (with roles) --> DepChainTagger --> Output relation layer with matches
```


Let's start with a simple pattern that finds a root and its direct children in the syntax layer. We will define a pattern with two nodes: the first node is the root of the sentence, and the second node is any child of the root. We will label the root as `root` and the child as `first_level_child` in the output relation layer. The edge constraint will specify that they can look only one step away from the root.


In [55]:
root_pattern = PathPattern(
    name="root_and_children",
    node_steps=(
        NodeConstraint(
            role="root",
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="root"),
                # "text": ValueCondition(mode=ConditionMode.WILDCARD),
                "upostag": ValueCondition(mode=ConditionMode.WILDCARD),
            },
        ),
        NodeConstraint(
            role="first_level_child",
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.DOWN,
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="obl")
            },
            min_hops=1,
            max_hops=1,
        ),
    ),
)

Now, let's instantiate the tagger with this pattern and see how it works on our sample text.


In [56]:
verb_tagger = DepChainTagger(patterns=(root_pattern,), include_pattern_constraints=True)

In [57]:
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

Let's check the Text object after tagging.


In [58]:
display(text_obj)

Text(text="1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.")

There is a new relation layer named `dep_chains` that contains the matches found by the tagger. Each match is a relation with the roles defined by the pattern (for example `root` and `first_level_child` in the simple pattern above). The relation also contains several attributes:

- One attribute for each role defined in the pattern, pointing to the matched node for that role.
- `pattern_name`: the name of the pattern that produced the match.
- `matched_text`: the surface text covered by the matched nodes.

If you enabled `include_pattern_constraints=True` when constructing the tagger, as we did here, additional attributes derived from the pattern's constraints (prefixed with the role name) will also be included in the output relation. The exact number and content of matches depends on the quality of the syntax parse and the Stanza model version used in your environment.


Let's inspect the first match in more detail to see what information is available.


In [59]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('root', 'first_level_child'), attributes=('pattern_name', 'matched_text', 'root_deprel', 'root_upostag'), relations=[Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'suvel')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas suvel', 'root_deprel': <class 'dict'>, 'root_upostag': 'WILDCARD'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'laeva')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas laeva', 'root_deprel': <class 'dict'>, 'root_upostag': 'WILDCARD'}])])

1990. aasta kuumal suvel first_level_child(0) vaatas root(0), root(1) Bureau Veritas Estline'i omanduseks saanud laeva first_level_child(1) uuesti üle.

In [60]:
print(text_obj["dep_chains"][0])

Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'suvel')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas suvel', 'root_deprel': <class 'dict'>, 'root_upostag': 'WILDCARD'}])


The output layer is a relation layer (default name `dep_chains`), so each match is represented as a relation whose role fields point to the matched syntax nodes/spans (e.g. `root`, `first_level_child`). Each relation also contains standard attributes: `pattern_name` (the producing pattern) and `matched_text` (surface text covered by the matched nodes). If `include_pattern_constraints=True` was set when constructing the tagger, additional attributes derived from the pattern constraints are included, prefixed with the role name (for example `verb_upostag`).

In the example above:

- The `root` attribute points to the root's `role` we defined in the pattern.
- The `first_level_child` attribute points to the child node's `role` we defined in the pattern.
- The `pattern_name` attribute indicates which pattern was matched.
- The `matched_text` attribute shows the text covered by the matched nodes.
- The `root_deprel` and `root_upostag` attributes (included because we set `include_pattern_constraints=True`) show the constraints defined for the `root` role in the pattern.


## Going in-depth on patterns and tagger configuration


### Path patterns


The `PathPattern` class is the core of the pattern-matching system. It defines a sequence of node and edge constraints (`NodeConstraint` and `EdgeConstraint` respectively) that describe the structure we want to match in the dependency tree. Each node constraint specifies a role and conditions on the node's attributes or features, while each edge constraint specifies the direction and conditions on the dependency relation between nodes.

Note that constraints on the edge are directed to the node it connects to, so they describe the relationship from the perspective of the node at the end of the edge.

The tagger can take multiple `PathPattern` objects, allowing you to search for different structures in the same run. Each pattern can have a name, and the output relations will indicate which pattern produced each match.

Let's look at the two main types of constraints you can use: `NodeConstraint` and `EdgeConstraint`.


### NodeConstraints


`NodeConstraint` objects specify conditions on the nodes in the pattern. They can check for specific attribute values, the presence of certain attributes, or more complex nested checks using `NestedValueCondition`.

**Key parameters**

- `role`: The role name for the node in the output relation layer.
- `attribute_conditions`: A dictionary where keys are scalar attribute names and values are condition objects such as `ValueCondition`. Use this for ordinary node attributes like `upostag`, `deprel`, `lemma`, `id`, or `head`. Example: `{"upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")}`
- `nested_attribute_conditions`: A dictionary where keys are names of nested or dictionary-like attributes (for example, `feats`) and values are `NestedValueCondition` objects used to test those nested structures. Example: `{"feats": NestedValueCondition(required={"Number": "Sing"})}`
- `extra_predicates`: An optional sequence of callables `predicate(node) -> bool` that perform arbitrary checks on candidate nodes. All predicates must return `True` for the node to satisfy the constraint.

Use `attribute_conditions` for scalar attributes (simple equality/regex/membership checks via `ValueCondition`) and `nested_attribute_conditions` for nested/dict-like data (use `NestedValueCondition` to express required/forbidden sub-structures). This separation helps keep patterns clear and avoids mixing scalar and nested matching semantics.

---


Let's create a simple `NodeConstraint` that matches a verb whose `feats` is `s`. We can use `attribute_conditions` for the `upostag` to specify that it must be a verb (upostag `V`), and we can use `nested_attribute_conditions` for the `feats` to specify that it must be third person singular (`ps3` and `sg`)


In [20]:
pattern_1 = PathPattern(
    name="node_constraint",
    node_steps=(
        NodeConstraint(
            role="verb_s",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")
            },
            nested_attribute_conditions={
                "feats": NestedValueCondition(
                    mode=ConditionMode.EXACT,
                    required={
                        "ps3": "ps3",
                        "sg": "sg",
                    },
                    allow_extra_keys=True,  # allowing extra keys means we only require the presence of ps3 and sg, but other features can also be present without causing a mismatch
                ),
            },
        ),
    ),
    edge_steps=(),
)

In [21]:
# Tagger instantiation
dct = DepChainTagger(patterns=(pattern_1,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
dct.tag(text_obj);

In [22]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('verb_s',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(verb_s: 'vaatas')], [{'pattern_name': 'node_constraint', 'matched_text': 'vaatas'}])])

1990. aasta kuumal suvel vaatas verb_s(0) Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.

We see that word "vaatas" fits the constraints we defined: it has `upostag` `V` and its `feats` include `ps3` and `sg`.

---


### EdgeConstraints


`EdgeConstraint` objects specify the relationship between consecutive `NodeConstraint` steps in a `PathPattern`. Edge attribute checks test the dependency relation connecting the two nodes, not a node's own attributes.

**Key parameters**

- `direction`: `DirectionMode.DOWN` or `DirectionMode.UP`. `DOWN` moves from a node to its child(ren); `UP` moves to the parent. Direction controls how the matcher traverses the `stanza_syntax` tree.
- `min_hops`, `max_hops`: Integers controlling how many dependency hops may be traversed for this edge. Use `min_hops=max_hops=1` for a single immediate relation; larger ranges allow skipping intermediate nodes.
- `attribute_conditions`: Mapping of scalar edge attributes → `ValueCondition` (e.g. `{'deprel': ValueCondition(mode=ConditionMode.EXACT, value='obl')}`). Use `ValueCondition` for scalar checks (including `REGEX`).
- `nested_attribute_conditions`: Mapping of nested/dict-like edge attributes → `NestedValueCondition` for complex checks on nested structures. This is useful for attributes like `feats` that contain multiple sub-features; you can specify required or forbidden sub-features without needing to match the entire structure.
- `extra_predicates`: Optional callables `predicate(parent_node, child_node, edge_meta) -> bool` for arbitrary edge-level checks. All predicates must return `True` for the edge to satisfy the constraint.

**Semantics & tips**

- Relation attributes (for example `deprel`) are taken from the child node's perspective, regardless of the edge direction. This means that even if you are traversing "up" from a child to its parent, the edge attributes you check will be those that describe the child's relation to its parent (for example, the child's `deprel`), not attributes of the parent node.
  - Illustration of edge direction and perspective:

  ```
  NodeConstraint A ---------------> EdgeConstraint A ---------------> NodeConstraint B
    (A's role)          (A's conditions on the edge from A to B)         (B's role)

  NodeConstraint A <--------------- EdgeConstraint B <--------------- NodeConstraint B
    (A's role)          (B's conditions on the edge from B to A)         (B's role)
  ```

  - These same edge constraints can be applied directly to the node they point to, but placing them in `EdgeConstraint` emphasizes that they are conditions on the relationship between nodes rather than on the node itself.

- Prefer precise `min_hops`/`max_hops` and `attribute_conditions` to improve matching speed and reduce spurious matches.
- An empty `EdgeConstraint` is most efficient; use `WILDCARD` to document intent when you want any edge value but also record the constraint (useful when `include_pattern_constraints=True`).
- `DepChildTagger` enforces that all edges are `DirectionMode.DOWN` (it matches child-only relations); using `UP` edges will raise a `ValueError` during validation.

---


Let's make the pattern more complex by changing it from matching a node to matching a chain of two nodes: a root and its child. We will need to define two `NodeConstraint` objects (one for the root and one for the child) and an `EdgeConstraint` to connect them.

We create three patterns with the same node constraints but different edge constraints to illustrate how edge conditions affect the matches. The first pattern requires an immediate child (`min_hops=1, max_hops=1`), the second allows descendants at four levels of separation (`min_hops=1, max_hops=4`), and the third allows descendants that are not direct children but still within four hops (`min_hops=2, max_hops=4`).


In [10]:
pattern_1 = PathPattern(
    name="root_and_obl_children",
    node_steps=(
        NodeConstraint(
            role="root",
            attribute_conditions={
                "text": ValueCondition(mode=ConditionMode.EXACT, value="vaatas")
            },
        ),
        NodeConstraint(
            role="child",
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.DOWN,
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="obl")
            },
            min_hops=1,
            max_hops=4,
        ),
    ),
)

pattern_2 = PathPattern(
    name="root_and_obl_direct_children",
    node_steps=(
        NodeConstraint(
            role="root",
            attribute_conditions={
                "text": ValueCondition(mode=ConditionMode.EXACT, value="vaatas")
            },
        ),
        NodeConstraint(
            role="child",
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.DOWN,
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="obl")
            },
            min_hops=1,
            max_hops=1,
        ),
    ),
)

pattern_3 = PathPattern(
    name="root_and_obl_not_direct_children",
    node_steps=(
        NodeConstraint(
            role="root",
            attribute_conditions={
                "text": ValueCondition(mode=ConditionMode.EXACT, value="vaatas")
            },
        ),
        NodeConstraint(
            role="child",
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.DOWN,
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="obl")
            },
            min_hops=2,
            max_hops=4,
        ),
    ),
)

In [11]:
# Tagger instantiation
dct = DepChainTagger(patterns=(pattern_1, pattern_2, pattern_3))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
dct.tag(text_obj);

In [12]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('root', 'child'), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(root: 'vaatas'), NamedSpan(child: 'suvel')], [{'pattern_name': 'root_and_obl_children', 'matched_text': 'vaatas suvel'}, {'pattern_name': 'root_and_obl_direct_children', 'matched_text': 'vaatas suvel'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(child: 'laeva')], [{'pattern_name': 'root_and_obl_children', 'matched_text': 'vaatas laeva'}, {'pattern_name': 'root_and_obl_direct_children', 'matched_text': 'vaatas laeva'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(child: "Estline'i")], [{'pattern_name': 'root_and_obl_children', 'matched_text': "vaatas Estline'i"}, {'pattern_name': 'root_and_obl_not_direct_children', 'matched_text': "vaatas Estline'i"}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(child: 'omanduseks')], [{'pattern_name': 'root_and_obl_children', 'matched_text': 'vaatas omanduseks'}, {'pattern_name': 'root_and_obl_not_direct_children', 'matched_text': 'vaatas omanduseks'}])])

1990. aasta kuumal suvel child(0) vaatas root(0), root(1), root(2), root(3) Bureau Veritas Estline'i child(2) omanduseks child(3) saanud laeva child(1) uuesti üle.

We see that the first pattern matches all children of the root with the specified node constraints. The second pattern matches only the immediate children of the root, while the third pattern matches only the more distant descendants (not direct children) that still satisfy the node constraints. This illustrates how edge constraints can be used to control the structural relationships between matched nodes in a `PathPattern`.

---


### Condition types


There are two condition types you will see in pattern definitions: `ValueCondition` and `NestedValueCondition`.

`ValueCondition`

`ValueCondition` is the default choice for simple scalar attributes such as `upostag`, `deprel`, `lemma`, `id`, or `head`. Use it whenever the attribute you are checking is a single value (string, number, etc.). `ValueCondition` supports the standard condition modes: `EXACT`, `NEGATION`, `WILDCARD`, `MEMBERSHIP`, `NOT_MEMBERSHIP`, and `REGEX` (for text-based pattern matching).

Typical `ValueCondition` examples:

- check that a node is a verb: `{"upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")}`
- allow verbs or nouns: `{"upostag": ValueCondition(mode=ConditionMode.MEMBERSHIP, value=["V", "S"])}`
- match by regular expression: `{"deprel": ValueCondition(mode=ConditionMode.REGEX, value="^obj.*")}`

`NestedValueCondition`

`NestedValueCondition` is intended for nested, dictionary-like or collection-valued attributes — most commonly the morphological `feats` map. Instead of comparing a single scalar, it lets you assert the presence or absence of sub-keys and sub-values using `required` and `forbidden` specifications (and related options such as `allow_extra_keys`). `NestedValueCondition` is designed for structured matching and does not provide scalar `REGEX` semantics; use `ValueCondition` for regex checks on simple attributes.

Typical `NestedValueCondition` examples:

- require a singular noun inside `feats`: `{"feats": NestedValueCondition(required={"Number": "Sing"})}`
- require several features and forbid another: `NestedValueCondition(required={"Number": "Pl"}, forbidden={"Case": "Gen"})`

Practical rule of thumb

- Use `ValueCondition` for simple, direct attributes on nodes or edges (scalar values, regex, membership checks).
- Use `NestedValueCondition` for nested/dict-like attribute information (for example `feats`) when you need to assert sub-keys or sub-values.

This separation keeps patterns readable and prevents mixing scalar and nested matching semantics.


### Condition modes


The meaning of a condition mode depends on the condition class that uses it.

#### `ValueCondition` for scalar attributes

`ValueCondition` is the right choice for simple attributes that hold one value, such as `upostag`, `lemma`, `deprel`, `id`, or `head`. Here the mode controls how one scalar actual value is compared against one expected value.

- `EXACT`: the actual value must be equal to the expected value.
- `NEGATION`: the actual value must be different from the expected value.
- `WILDCARD`: any actual value is accepted; the expected value must be `None`.
- `MEMBERSHIP`: the actual value must be one of the values in the expected iterable.
- `NOT_MEMBERSHIP`: the actual value must not be one of the values in the expected iterable.
- `REGEX`: the text form of the actual value must match the expected regular expression.

Typical examples:

- `upostag=ValueCondition(mode=ConditionMode.EXACT, value="V")` matches only verbs.
- `upostag=ValueCondition(mode=ConditionMode.MEMBERSHIP, value=["V", "S"])` matches verbs and nouns.
- `deprel=ValueCondition(mode=ConditionMode.REGEX, value="^obj.*")` matches dependency labels that start with `obj`.

#### `NestedValueCondition` for nested or dictionary-like attributes

`NestedValueCondition` is used for attributes whose values are nested structures, most commonly the morphological `feats` map. Instead of comparing one scalar value, it compares nested content such as dictionaries, lists, tuples, sets, or combinations of them.

The modes are interpreted differently here:

- `EXACT`: the actual nested structure must contain the required content, and it must not violate the forbidden content. If `allow_extra_keys=False`, no additional keys are allowed beyond those specified by the condition.
- `NEGATION`: the actual nested structure must not match the required content, and it must also avoid the forbidden content.
- `WILDCARD`: any nested value is accepted.
- `MEMBERSHIP`: at least one candidate in `required` must be found somewhere in the nested structure; `forbidden` still blocks a match.

Typical examples:

- `nested_attribute_conditions={"feats": NestedValueCondition(mode=ConditionMode.EXACT, required={"Number": "Sing"})}` matches nodes whose `feats` contain singular number.
- `nested_attribute_conditions={"feats": NestedValueCondition(mode=ConditionMode.EXACT, required={"Case": "Gen"}, forbidden={"Number": "Plur"})}` matches nodes whose `feats` contain genitive case and do not contain plural number.

#### Practical rule of thumb

- Use `ValueCondition` for ordinary scalar values.
- Use `NestedValueCondition` for nested, dictionary-like, or collection-valued attributes.

The same `ConditionMode` name can therefore behave differently depending on the condition class: in `ValueCondition` it compares one scalar value, while in `NestedValueCondition` it evaluates the shape and content of a nested structure.

This separation keeps pattern definitions readable and makes the intended matching behaviour explicit.


Let's take the example above and modify the pattern to use different condition modes to see how it affects the matches and the output layer. For the brevity of the example, let's define all the patterns at once.


#### Using `EXACT` mode


First is condition mode `EXACT` that we have already seen. There are two patterns:

- One that matches only verbs with `upostag = V`.
- The other matches only verbs with `upostag = V` and third person singular `feats` (`ps3` and `sg`).


In [ ]:
# Pattern definition using EXACT condition mode
v_pattern_exact = PathPattern(
    name="exact_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="verb",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")
            },
        ),
    ),
    edge_steps=(),
)
v_pattern_n_exact = PathPattern(
    name="exact_verb_pattern_nested",
    node_steps=(
        NodeConstraint(
            role="verb",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")
            },
            nested_attribute_conditions={
                "feats": NestedValueCondition(
                    mode=ConditionMode.EXACT,
                    required={
                        "ps3": "ps3",
                        "sg": "sg",
                    },
                    allow_extra_keys=True,
                )
            },
        ),
    ),
    edge_steps=(),
)

In [66]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_exact, v_pattern_n_exact))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [67]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('verb',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(verb: 'vaatas')], [{'pattern_name': 'exact_verb_pattern', 'matched_text': 'vaatas'}, {'pattern_name': 'exact_verb_pattern_nested', 'matched_text': 'vaatas'}]), Relation([NamedSpan(verb: 'saanud')], [{'pattern_name': 'exact_verb_pattern', 'matched_text': 'saanud'}])])

1990. aasta kuumal suvel vaatas verb(0) Bureau Veritas Estline'i omanduseks saanud verb(1) laeva uuesti üle.

The simpler (without nested conditions) pattern matches both _vaatas_ and _saanud_, while the more complex pattern with nested conditions matches only _vaatas_ because it is the only verb that has `feats` indicating third person singular.


#### Using `NEGATION` mode


Second is condition mode `NEGATION`. There are two patterns:

- One that matches any node that is not a verb (negating `upostag = V`).
- The other matches any node that is not a verb with third person singular `feats` (negating both `upostag = V` and the nested `feats` condition).


In [ ]:
# Pattern definition using NEGATED condition mode
v_pattern_negated = PathPattern(
    name="negation_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="not_verb",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.NEGATION, value="V")
            },
        ),
    ),
    edge_steps=(),
)
v_pattern_n_negated = PathPattern(
    name="negation_verb_pattern_nested",
    node_steps=(
        NodeConstraint(
            role="not_verb",
            nested_attribute_conditions={
                "feats": NestedValueCondition(
                    mode=ConditionMode.NEGATION,
                    required={
                        "ps3": "ps3",
                        "sg": "sg",
                    },
                    allow_extra_keys=True,
                )
            },
        ),
    ),
    edge_steps=(),
)

In [75]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_negated, v_pattern_n_negated))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [76]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('not_verb',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(not_verb: '1990.')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': '1990.'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': '1990.'}]), Relation([NamedSpan(not_verb: 'aasta')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'aasta'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'aasta'}]), Relation([NamedSpan(not_verb: 'kuumal')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'kuumal'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'kuumal'}]), Relation([NamedSpan(not_verb: 'suvel')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'suvel'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'suvel'}]), Relation([NamedSpan(not_verb: 'Bureau')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'Bureau'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'Bureau'}]), Relation([NamedSpan(not_verb: 'Veritas')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'Veritas'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'Veritas'}]), Relation([NamedSpan(not_verb: "Estline'i")], [{'pattern_name': 'negation_verb_pattern', 'matched_text': "Estline'i"}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': "Estline'i"}]), Relation([NamedSpan(not_verb: 'omanduseks')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'omanduseks'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(not_verb: 'laeva')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'laeva'}, {'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'laeva'}]), Relation([NamedSpan(not_verb: 'uuesti')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'uuesti'}]), Relation([NamedSpan(not_verb: 'üle')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'üle'}]), Relation([NamedSpan(not_verb: '.')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': '.'}]), Relation([NamedSpan(not_verb: 'saanud')], [{'pattern_name': 'negation_verb_pattern_nested', 'matched_text': 'saanud'}])])

1990. not_verb(0) aasta not_verb(1) kuumal not_verb(2) suvel not_verb(3) vaatas Bureau not_verb(4) Veritas not_verb(5) Estline'i not_verb(6) omanduseks not_verb(7) saanud not_verb(12) laeva not_verb(8) uuesti not_verb(9) üle not_verb(10) . not_verb(11)

With `NEGATION` mode, we get matches for all nodes that do not have `upostag` equal to `V`. The first pattern matches all non-verbs, while the second pattern matches all nodes that are not with third person singular `feats`, which will include _saanud_ because it does not have `feats` indicating third person singular (first pattern would exclude it because it is a verb).


#### Using `WILDCARD` mode


Third condition mode is `WILDCARD`, which matches any node regardless of its attributes.


In [37]:
# Pattern definition using WILDCARD condition mode
v_pattern_wildcard = PathPattern(
    name="wildcard_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="all",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.WILDCARD, value=None)
            },
        ),
    ),
    edge_steps=(),
)
v_pattern_n_wildcard = PathPattern(
    name="wildcard_verb_pattern_nested",
    node_steps=(
        NodeConstraint(
            role="all",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.WILDCARD, value=None)
            },
            nested_attribute_conditions={
                "feats": NestedValueCondition(
                    mode=ConditionMode.WILDCARD, required=None, allow_extra_keys=True
                )
            },
        ),
    ),
    edge_steps=(),
)

In [38]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_wildcard, v_pattern_n_wildcard))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [39]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('all',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(all: '1990.')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': '1990.'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': '1990.'}]), Relation([NamedSpan(all: 'aasta')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'aasta'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'aasta'}]), Relation([NamedSpan(all: 'kuumal')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'kuumal'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'kuumal'}]), Relation([NamedSpan(all: 'suvel')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'suvel'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'suvel'}]), Relation([NamedSpan(all: 'vaatas')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'vaatas'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'vaatas'}]), Relation([NamedSpan(all: 'Bureau')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'Bureau'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'Bureau'}]), Relation([NamedSpan(all: 'Veritas')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'Veritas'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'Veritas'}]), Relation([NamedSpan(all: "Estline'i")], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': "Estline'i"}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': "Estline'i"}]), Relation([NamedSpan(all: 'omanduseks')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'omanduseks'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(all: 'saanud')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'saanud'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'saanud'}]), Relation([NamedSpan(all: 'laeva')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'laeva'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'laeva'}]), Relation([NamedSpan(all: 'uuesti')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'uuesti'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'uuesti'}]), Relation([NamedSpan(all: 'üle')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'üle'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': 'üle'}]), Relation([NamedSpan(all: '.')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': '.'}, {'pattern_name': 'wildcard_verb_pattern_nested', 'matched_text': '.'}])])

1990. all(0) aasta all(1) kuumal all(2) suvel all(3) vaatas all(4) Bureau all(5) Veritas all(6) Estline'i all(7) omanduseks all(8) saanud all(9) laeva all(10) uuesti all(11) üle all(12) . all(13)

As expected, with `WILDCARD` mode we get matches for all nodes in the syntax layer.


#### Using `MEMBERSHIP` mode


The fourth condition mode is `MEMBERSHIP`, which matches nodes whose `upostag` value is either `V` or `S`. There are two patterns:

- One that matches nodes with `upostag` in `["V", "S"]`.
- The other matches nodes with `upostag` in `["V", "S"]`, but also requires that if the node has `feats`, it must include either `ps3` or `sg` (third person or singular).


In [ ]:
# Pattern definition using MEMBERSHIP condition mode
v_and_s_pattern_membership = PathPattern(
    name="membership_verb_and_noun_pattern",
    node_steps=(
        NodeConstraint(
            role="verb_or_noun",
            attribute_conditions={
                "upostag": ValueCondition(
                    mode=ConditionMode.MEMBERSHIP, value=["V", "S"]
                )
            },
        ),
    ),
    edge_steps=(),
)
v_and_s_pattern_n_membership = PathPattern(
    name="membership_verb_and_noun_pattern_nested",
    node_steps=(
        NodeConstraint(
            role="verb_or_noun",
            attribute_conditions={
                "upostag": ValueCondition(
                    mode=ConditionMode.MEMBERSHIP, value=["V", "S"]
                )
            },
            nested_attribute_conditions={
                "feats": NestedValueCondition(
                    mode=ConditionMode.MEMBERSHIP,
                    required={
                        "ps3": "ps3",
                        "sg": "sg",
                    },
                    allow_extra_keys=True,
                )
            },
        ),
    ),
    edge_steps=(),
)

In [80]:
# Tagger instantiation
verb_tagger = DepChainTagger(
    patterns=(v_and_s_pattern_membership, v_and_s_pattern_n_membership)
)

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [81]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('verb_or_noun',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(verb_or_noun: 'aasta')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'aasta'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'aasta'}]), Relation([NamedSpan(verb_or_noun: 'kuumal')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'kuumal'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'kuumal'}]), Relation([NamedSpan(verb_or_noun: 'suvel')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'suvel'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'suvel'}]), Relation([NamedSpan(verb_or_noun: 'vaatas')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'vaatas'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'vaatas'}]), Relation([NamedSpan(verb_or_noun: 'Bureau')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'Bureau'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'Bureau'}]), Relation([NamedSpan(verb_or_noun: 'Veritas')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'Veritas'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'Veritas'}]), Relation([NamedSpan(verb_or_noun: "Estline'i")], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': "Estline'i"}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': "Estline'i"}]), Relation([NamedSpan(verb_or_noun: 'omanduseks')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'omanduseks'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(verb_or_noun: 'saanud')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'saanud'}]), Relation([NamedSpan(verb_or_noun: 'laeva')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'laeva'}, {'pattern_name': 'membership_verb_and_noun_pattern_nested', 'matched_text': 'laeva'}])])

1990. aasta verb_or_noun(0) kuumal verb_or_noun(1) suvel verb_or_noun(2) vaatas verb_or_noun(3) Bureau verb_or_noun(4) Veritas verb_or_noun(5) Estline'i verb_or_noun(6) omanduseks verb_or_noun(7) saanud verb_or_noun(8) laeva verb_or_noun(9) uuesti üle.

The first pattern matches all verbs and nouns. The second pattern is almost the same, because all nouns have `sg` in their `feats` if they are singular. However, it excludes _saanud_ as it does not have `feats` indicating third person nor singular, while _vaatas_ does (has both `ps3` and `sg`).


#### Using `NOT_MEMBERSHIP` mode


The fourth condition mode is `NOT_MEMBERSHIP`, which matches nodes whose `upostag` value is neither `V` nor `S`. This will match all nodes except verbs and nouns.


In [ ]:
# Pattern definition using NOT_MEMBERSHIP condition mode
v_and_s_pattern_not_membership = PathPattern(
    name="not_membership_verb_and_noun_pattern",
    node_steps=(
        NodeConstraint(
            role="not_verb_or_noun",
            attribute_conditions={
                "upostag": ValueCondition(
                    mode=ConditionMode.NOT_MEMBERSHIP, value=["V", "S"]
                )
            },
        ),
    ),
    edge_steps=(),
)

In [59]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_and_s_pattern_not_membership,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [60]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('not_verb_or_noun',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(not_verb_or_noun: '1990.')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': '1990.'}]), Relation([NamedSpan(not_verb_or_noun: 'Bureau')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'Bureau'}]), Relation([NamedSpan(not_verb_or_noun: 'Veritas')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'Veritas'}]), Relation([NamedSpan(not_verb_or_noun: "Estline'i")], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': "Estline'i"}]), Relation([NamedSpan(not_verb_or_noun: 'saanud')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'saanud'}]), Relation([NamedSpan(not_verb_or_noun: 'uuesti')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'uuesti'}]), Relation([NamedSpan(not_verb_or_noun: 'üle')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'üle'}]), Relation([NamedSpan(not_verb_or_noun: '.')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': '.'}])])

1990. not_verb_or_noun(0) aasta kuumal suvel vaatas Bureau not_verb_or_noun(1) Veritas not_verb_or_noun(2) Estline'i not_verb_or_noun(3) omanduseks saanud not_verb_or_noun(4) laeva uuesti not_verb_or_noun(5) üle not_verb_or_noun(6) . not_verb_or_noun(7)

All nodes with `upostag` not equal to `V` or `S` are matched.


#### Using `REGEX` mode


The final condition mode is `REGEX`, which matches nodes whose `upostag` value starts with `V`. This will match all verbs, including those with more specific tags like `VERB`.


In [ ]:
# Pattern definition using REGEX condition mode
v_pattern_regex = PathPattern(
    name="regex_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="starts_with_v",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.REGEX, value="^V.*")
            },
        ),
    ),
    edge_steps=(),
)

In [61]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_regex,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [62]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('starts_with_v',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(starts_with_v: 'vaatas')], [{'pattern_name': 'regex_verb_pattern', 'matched_text': 'vaatas'}])])

1990. aasta kuumal suvel vaatas starts_with_v(0) Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.

Since all verbs in the syntax layer have `upostag` values that start with `V`, we get the same matches as with `EXACT` mode.


### Deduplication modes


Deduplication is primarily a defensive and future-facing feature. For the current sentence-tree matcher identical role→token assignments are rare, but duplicates can still arise in other situations (for example: duplicate lower-level inputs, intentionally repeated patterns in tests, non-tree graphs with multiple traversal paths, or when matches are merged across runs).

The available modes give you control over how aggressively to collapse similar matches:

- `none`: keep all matches (useful for exhaustive output or debugging).
- `exact`: collapse only fully identical matches (same role assignments and same `matched_text`).
- `role_based`: collapse matches that assign the same token IDs to roles (ignores `matched_text`).

<!-- If you want pattern-agnostic collapsing (merge matches produced by different pattern names), apply a small post-processing step that groups results by the role token IDs, or opt to add a dedicated `pattern_agnostic` dedup mode in the library. -->
